# **Exercícios de Python utilizando o SQLite**
---
Nome: Giovanna Coelho Gilio de Sá
Turma: 2°Semestre - AA

### **1. Instalar e Importar bibliotecas**

In [1]:
!pip install requests
!pip install beautifulsoup4
!pip install sqlite3
!pip install pandas

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


In [2]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd

### **2. Configuração e Criação do Banco de Dados**

##### **Explicação:** O código abaixo faz a conexão com o banco existente, ou se são existir faz a criação do banco.

In [3]:
conn = sqlite3.connect('empresa.db')
cursor = conn.cursor()

#### **2.1 Criação de tabela DDL**

In [4]:
cursor.execute("""
  CREATE TABLE IF NOT EXISTS funcionarios(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cargo TEXT NOT NULL,
    salario REAL,
    data_contratacao TEXT NOT NULL
  );
""")
print("Tabela funcionarios criada")

Tabela funcionarios criada


In [5]:
cursor.execute("PRAGMA table_info(funcionarios);")
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'nome', 'TEXT', 1, None, 0)
(2, 'cargo', 'TEXT', 1, None, 0)
(3, 'salario', 'REAL', 0, None, 0)
(4, 'data_contratacao', 'TEXT', 1, None, 0)


### **3. Manipulação de Dados (DML)**

In [6]:
cursor.execute("""
  INSERT INTO funcionarios(nome, cargo, salario, data_contratacao) VALUES
  ('Ana Silva', 'Desenvolvedora', 7500.00, '2024-01-15'),
  ('Carlos Souza', 'Gerente de Projetos', 4200.50, '2023-05-20'),
  ('Mariana Lima', 'Gerente de Projetos', 11000.00, '2022-11-10'),
  ('João Santos', 'Assistente Administrativo', 2800.00, '2025-02-01'),
  ('Beatriz Costa', 'Designer UX', 6200.00, '2024-08-12'),
  ('Yugo Luiz', 'Designer UX', 3500.00, '2025-02-18');
""")
conn.commit()

##### Exibir todos os registros da tabela funcionarios

In [7]:
cursor.execute("SELECT * FROM funcionarios;")
cursor.fetchall()

[(1, 'Ana Silva', 'Desenvolvedora', 7500.0, '2024-01-15'),
 (2, 'Carlos Souza', 'Gerente de Projetos', 4200.5, '2023-05-20'),
 (3, 'Mariana Lima', 'Gerente de Projetos', 11000.0, '2022-11-10'),
 (4, 'João Santos', 'Assistente Administrativo', 2800.0, '2025-02-01'),
 (5, 'Beatriz Costa', 'Designer UX', 6200.0, '2024-08-12'),
 (6, 'Yugo Luiz', 'Designer UX', 3500.0, '2025-02-18')]

##### Exibir apenas os funcionários com salário superior a R$ 5.000.

In [8]:
cursor.execute("""
  SELECT * FROM funcionarios
  WHERE salario > 5000;
""")
cursor.fetchall()

[(1, 'Ana Silva', 'Desenvolvedora', 7500.0, '2024-01-15'),
 (3, 'Mariana Lima', 'Gerente de Projetos', 11000.0, '2022-11-10'),
 (5, 'Beatriz Costa', 'Designer UX', 6200.0, '2024-08-12')]

##### Exibir os funcionários que trabalham como 'Gerente de Projetos'


In [9]:
cursor.execute("""
  SELECT * FROM funcionarios
  WHERE cargo = 'Gerente de Projetos';
""")
cursor.fetchall()

[(2, 'Carlos Souza', 'Gerente de Projetos', 4200.5, '2023-05-20'),
 (3, 'Mariana Lima', 'Gerente de Projetos', 11000.0, '2022-11-10')]

##### Atualização de dados (UPDATE): Atualizar o salário de um funcionário específico.

In [10]:
cursor.execute("""
  UPDATE funcionarios SET salario = 8000 WHERE nome = "Ana Silva";
""")
conn.commit()

In [11]:
cursor.execute("SELECT * FROM funcionarios;")
cursor.fetchall()

[(1, 'Ana Silva', 'Desenvolvedora', 8000.0, '2024-01-15'),
 (2, 'Carlos Souza', 'Gerente de Projetos', 4200.5, '2023-05-20'),
 (3, 'Mariana Lima', 'Gerente de Projetos', 11000.0, '2022-11-10'),
 (4, 'João Santos', 'Assistente Administrativo', 2800.0, '2025-02-01'),
 (5, 'Beatriz Costa', 'Designer UX', 6200.0, '2024-08-12'),
 (6, 'Yugo Luiz', 'Designer UX', 3500.0, '2025-02-18')]

##### Exclusão de dados (DELETE): Exclui um dos funcionarios

In [12]:
# Quantidade de registros antes da exclusão
cursor.execute("SELECT COUNT(*) FROM funcionarios;")
cursor.fetchone()

(6,)

In [13]:
cursor.execute("""
  DELETE FROM funcionarios WHERE nome = "João Santos";
""")
conn.commit()

In [14]:
# Quantidade de registros pós exclusão
cursor.execute("SELECT COUNT(*) FROM funcionarios;")
cursor.fetchone()

(5,)

### **4. Função de Agregação e Ordenação**

##### Soma total dos salários

In [15]:
cursor.execute("""
  SELECT SUM(salario) FROM funcionarios;
""")
cursor.fetchone()

(32900.5,)

##### Salário médio da empresa

In [16]:
cursor.execute("""
  SELECT AVG(salario) FROM funcionarios;
""")
cursor.fetchone()

(6580.1,)

Salario mais alto e o mais baixo

In [17]:
cursor.execute("""
  SELECT
    MAX(salario),
    MIN(salario)
    FROM funcionarios;
""")
cursor.fetchone()

(11000.0, 3500.0)

##### Agrupamento de dados (GROUP BY)

##### Calcule um salário médio para cada cargo.


In [18]:
cursor.execute("""
  SELECT avg(salario) FROM funcionarios
  GROUP BY cargo
""")
cursor.fetchall()

[(8000.0,), (4850.0,), (7600.25,)]

##### Conte quantos funcionários existem em cada cargo.

In [19]:
cursor.execute("""
  SELECT count(id) FROM funcionarios
  GROUP BY cargo
""")
cursor.fetchall()

[(1,), (2,), (2,)]

##### Ordenação (ORDER BY)

##### Selecione todos os funcionários e ordens de maior para o menor salário.


In [20]:
cursor.execute("""
  SELECT * FROM funcionarios
  ORDER BY salario DESC;
""")
cursor.fetchall()

[(3, 'Mariana Lima', 'Gerente de Projetos', 11000.0, '2022-11-10'),
 (1, 'Ana Silva', 'Desenvolvedora', 8000.0, '2024-01-15'),
 (5, 'Beatriz Costa', 'Designer UX', 6200.0, '2024-08-12'),
 (2, 'Carlos Souza', 'Gerente de Projetos', 4200.5, '2023-05-20'),
 (6, 'Yugo Luiz', 'Designer UX', 3500.0, '2025-02-18')]

##### Selecione todos os funcionários e ordene-os por nome em ordem alfabética.

In [21]:
cursor.execute("""
  SELECT nome FROM funcionarios
  ORDER BY nome ASC;
""")
cursor.fetchall()

[('Ana Silva',),
 ('Beatriz Costa',),
 ('Carlos Souza',),
 ('Mariana Lima',),
 ('Yugo Luiz',)]

### **5. Transações e ROLLBACK**

##### Simulação de transação

In [22]:
try:
  conn.execute("BEGIN;")
  cursor.execute("""INSERT INTO funcionarios(nome, cargo, salario, data_contratacao)
  VALUES(
    "Giovanna Coelho",
    "Desenvolvedora",
    10200,
    "12-08-2026"
    );""")
  raise Exception("Erro simulado!")
  conn.commit()
except:
  conn.rollback()
  print("Transação revertida")


Transação revertida


##### Transação com COMMIT

In [23]:
try:
  conn.execute("BEGIN;")
  cursor.execute("""INSERT INTO funcionarios(nome, cargo, salario, data_contratacao)
  VALUES(
    "Giovanna Coelho",
    "Desenvolvedora",
    10200,
    "12-08-2026"
    );""")
  conn.commit()
  print("Transação concluida")
except:
  conn.rollback()
  print("Transação revertida")


Transação concluida


In [24]:
conn.execute("SELECT * FROM funcionarios")
cursor.fetchall()

[]

### **6. Joins e Múltiplas Tabelas**

###### Criação de uma segunda tabela

In [25]:
cursor.execute("""
  CREATE TABLE IF NOT EXISTS departamentos(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_departamento TEXT NOT NULL,
    localizacao TEXT
  );
""")
print("Tabela departamentos criada")

Tabela departamentos criada


In [32]:
cursor.execute("""
  INSERT INTO departamentos (nome_departamento, localizacao) VALUES
  ('Design', 'Andar 10'),
  ('TI', 'Andar 4'),
  ('Adiministração', 'Andar 1');
""")
conn.commit()

In [36]:
cursor.execute("ALTER TABLE funcionarios ADD COLUMN departamento_id INTEGER REFERENCES departamentos(id);")
conn.commit()
print("Coluna departamento_id adicionada à tabela funcionarios")

Coluna departamento_id adicionada à tabela funcionarios


In [37]:
cursor.execute("""UPDATE funcionarios SET departamento_id = 1 WHERE id = 5""")
conn.commit()

In [38]:
cursor.execute("""UPDATE funcionarios SET departamento_id = 1 WHERE id = 6""")
conn.commit()

In [39]:
cursor.execute("""UPDATE funcionarios SET departamento_id = 2 WHERE id = 1""")
conn.commit()

In [40]:
cursor.execute("""UPDATE funcionarios SET departamento_id = 3 WHERE id = 4""")
conn.commit()

###### INNER JOIN

In [43]:
cursor.execute("""
    SELECT f.id, f.nome, d.nome_departamento AS departamentos
    FROM funcionarios f
    INNER JOIN departamentos d ON f.departamento_id = d.id;
""")
print(cursor.fetchall())

[(1, 'Ana Silva', 'TI'), (5, 'Beatriz Costa', 'Design'), (6, 'Yugo Luiz', 'Design')]


###### Criação de nova tabela projetos

In [46]:
cursor.execute("""
  CREATE TABLE IF NOT EXISTS projetos(
    projeto_id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_projeto TEXT NOT NULL
    );
""")
print("Tabela projetos criada")

Tabela projetos criada


###### Criação de nova tabela funcionarios projetos

In [45]:
cursor.execute("""
  CREATE TABLE IF NOT EXISTS funcionarios_projetos(
    funcionario_id INTEGER,
    projeto_id INTEGER,
    PRIMARY KEY (funcionario_id, projeto_id)
    FOREIGN KEY (projeto_id) REFERENCES projetos(id),
    FOREIGN KEY (funcionario_id) REFERENCES funcionarios(id)
    );
""")
print("Tabela funcionario_projetos criada")

Tabela funcionario_projetos criada


In [47]:
cursor.execute("""
  INSERT INTO projetos (nome_projeto) VALUES
  ('Redesign do Site'),
  ('Visita no escritório'),
  ('App Mobile');
""")
conn.commit()

In [48]:
cursor.execute("""
  INSERT INTO funcionarios_projetos (funcionario_id, projeto_id) VALUES
  (1, 1),
  (1, 2),
  (5, 1);
""")

conn.commit()

###### LEFT JOIN

In [49]:
cursor.execute("""
    SELECT
        f.nome AS funcionario,
        p.nome_projeto AS projeto
    FROM funcionarios f
    LEFT JOIN funcionarios_projetos fp ON f.id = fp.funcionario_id
    LEFT JOIN projetos p ON fp.projeto_id = p.id;
""")

for registro in cursor.fetchall():
    print(registro) # Professor descobri que esse for exibe melhor o fetch

('Ana Silva', 'Redesign do Site')
('Ana Silva', 'Visita no escritório')
('Carlos Souza', None)
('Mariana Lima', None)
('Beatriz Costa', 'Redesign do Site')
('Yugo Luiz', None)
('Giovanna Coelho', None)


### **Parte 7: DCL (Data Control Language)**

#### **Diferenças entre DDL, DML, DQL e DCL:**

* **DDL (Data Definition Language):** Utilizada para definir, altera e destroi estruturas do banco de dados (tabelas, índices, colunas).
  - Ex: `CREATE`, `ALTER`, `DROP`, `TRUNCATE`).
* **DML (Data Manipulation Language):** Utilizada para modificar e manipular os registros armazenados.
  - Ex: `INSERT`, `UPDATE`, `DELETE`.
* **DQL (Data Query Language):** Utilizada para consultar/buscar os dados armazenados no banco de dados.
  - Ex: `SELECT`.
* **DCL (Data Control Language):** Utilizada para controlar o acesso, privilégios e permissões de segurança dos usuários do banco de dados.
  - Ex.: `GRANT`, `REVOKE`

### **Parte 8: Encerrar banco de dados**

In [50]:
conn.close()
print("Conexao com o banco de dados encerrado!")

Conexao com o banco de dados encerrado!
